In [1]:
!pip install -q intake-esm gcsfs xarray zarr netCDF4 cftime dask nc-time-axis geopandas shapely pyogrio

In [2]:
import intake
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

cat_url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(cat_url)

col

,unique
activity_id,18
institution_id,36
source_id,88
experiment_id,170
member_id,657
table_id,37
variable_id,700
grid_label,10
zstore,514818
dcpp_init_year,61


In [3]:
cat = col.search(
    source_id="GFDL-ESM4",
    table_id="Amon",
    variable_id="tas",
    experiment_id=["historical", "ssp126", "ssp245", "ssp585"],
    member_id="r1i1p1f1"
)

cat.df[[
    "source_id", "experiment_id", "member_id",
    "table_id", "variable_id", "grid_label", "zstore"
]].drop_duplicates().sort_values(["experiment_id", "grid_label"])

,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore
3,GFDL-ESM4,historical,r1i1p1f1,Amon,tas,gr1,gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/hist...
1,GFDL-ESM4,ssp126,r1i1p1f1,Amon,tas,gr1,gs://cmip6/CMIP6/ScenarioMIP/NOAA-GFDL/GFDL-ES...
0,GFDL-ESM4,ssp245,r1i1p1f1,Amon,tas,gr1,gs://cmip6/CMIP6/ScenarioMIP/NOAA-GFDL/GFDL-ES...
2,GFDL-ESM4,ssp585,r1i1p1f1,Amon,tas,gr1,gs://cmip6/CMIP6/ScenarioMIP/NOAA-GFDL/GFDL-ES...


In [4]:
GRID_LABEL = cat.df["grid_label"].iloc[0]
print("Using grid_label:", GRID_LABEL)

cat = col.search(
    source_id="GFDL-ESM4",
    table_id="Amon",
    variable_id="tas",
    experiment_id=["historical", "ssp126", "ssp245", "ssp585"],
    member_id="r1i1p1f1",
    grid_label=GRID_LABEL
)

dsets = cat.to_dataset_dict(
    zarr_kwargs={"consolidated": True},
    storage_options={"token": "anon"}
)

for key in dsets.keys():
    print(key)

Using grid_label: gr1

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="4" value="4"></progress> 100.00% [4/4 00:02&lt;00:00]</div>

/usr/local/lib/python3.12/dist-packages/intake_esm/source.py:109: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 647. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/usr/local/lib/python3.12/dist-packages/intake_esm/source.py:109: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 647. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(url, **xarray_open_kwargs)


ScenarioMIP.NOAA-GFDL.GFDL-ESM4.ssp126.Amon.gr1
ScenarioMIP.NOAA-GFDL.GFDL-ESM4.ssp585.Amon.gr1
ScenarioMIP.NOAA-GFDL.GFDL-ESM4.ssp245.Amon.gr1
CMIP.NOAA-GFDL.GFDL-ESM4.historical.Amon.gr1


In [5]:
historical_ds = [ds for key, ds in dsets.items() if "historical" in key][0]
ssp126_ds = [ds for key, ds in dsets.items() if "ssp126" in key][0]
ssp245_ds = [ds for key, ds in dsets.items() if "ssp245" in key][0]
ssp585_ds = [ds for key, ds in dsets.items() if "ssp585" in key][0]

historical_ds

<xarray.Dataset> Size: 411MB
Dimensions:         (member_id: 1, dcpp_init_year: 1, time: 1980, lat: 180,
                     lon: 288, bnds: 2)
Coordinates:
  * member_id       (member_id) object 8B 'r1i1p1f1'
  * dcpp_init_year  (dcpp_init_year) object 8B None
  * time            (time) object 16kB 1850-01-16 12:00:00 ... 2014-12-16 12:...
  * lat             (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * lon             (lon) float64 2kB 0.625 1.875 3.125 ... 356.9 358.1 359.4
  * bnds            (bnds) float64 16B 1.0 2.0
    height          float64 8B ...
    lat_bnds        (lat, bnds) float64 3kB dask.array<chunksize=(180, 2), meta=np.ndarray>
    lon_bnds        (lon, bnds) float64 5kB dask.array<chunksize=(288, 2), meta=np.ndarray>
    time_bnds       (time, bnds) object 32kB dask.array<chunksize=(1980, 2), meta=np.ndarray>
Data variables:
    tas             (member_id, dcpp_init_year, time, lat, lon) float32 411MB dask.array<chunksize=(1, 1, 600, 180, 288), meta=np.ndarray>
Attributes: (12/60)
    Conventions:                      CF-1.7 CMIP-6.0 UGRID-1.0
    activity_id:                      CMIP
    branch_method:                    standard
    branch_time_in_child:             0.0
    branch_time_in_parent:            36500.0
    comment:                          <null ref>
    ...                               ...
    intake_esm_attrs:variable_id:     tas
    intake_esm_attrs:grid_label:      gr1
    intake_esm_attrs:zstore:          gs://cmip6/CMIP6/CMIP/NOAA-GFDL/GFDL-ES...
    intake_esm_attrs:version:         20190726
    intake_esm_attrs:_data_format_:   zarr
    intake_esm_dataset_key:           CMIP.NOAA-GFDL.GFDL-ESM4.historical.Amo...

In [6]:
for name, ds in {
    "historical": historical_ds,
    "ssp126": ssp126_ds,
    "ssp245": ssp245_ds,
    "ssp585": ssp585_ds
}.items():
    print(name, ds["tas"].shape, ds["time"].min().values, ds["time"].max().values)

historical (1, 1, 1980, 180, 288) 1850-01-16 12:00:00 2014-12-16 12:00:00
ssp126 (1, 1, 1032, 180, 288) 2015-01-16 12:00:00 2100-12-16 12:00:00
ssp245 (1, 1, 1032, 180, 288) 2015-01-16 12:00:00 2100-12-16 12:00:00
ssp585 (1, 1, 1032, 180, 288) 2015-01-16 12:00:00 2100-12-16 12:00:00


In [7]:
def subset_us(ds):
    lat_min, lat_max = 18, 72

    if float(ds.lon.max()) > 180:
        lon_min, lon_max = 190, 310
    else:
        lon_min, lon_max = -170, -50

    return ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

hist_us = subset_us(historical_ds)
ssp126_us = subset_us(ssp126_ds)
ssp245_us = subset_us(ssp245_ds)
ssp585_us = subset_us(ssp585_ds)

In [8]:
def fix_lon(ds):
    ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180))
    return ds.sortby("lon")

hist_us = fix_lon(hist_us)
ssp126_us = fix_lon(ssp126_us)
ssp245_us = fix_lon(ssp245_us)
ssp585_us = fix_lon(ssp585_us)

In [9]:
baseline = (
    hist_us["tas"]
    .sel(time=slice("1995-01-01", "2014-12-31"))
    .mean("time")
)
baseline

<xarray.DataArray 'tas' (member_id: 1, dcpp_init_year: 1, lat: 54, lon: 96)> Size: 21kB
dask.array<mean_agg-aggregate, shape=(1, 1, 54, 96), dtype=float32, chunksize=(1, 1, 54, 96), chunktype=numpy.ndarray>
Coordinates:
  * member_id       (member_id) object 8B 'r1i1p1f1'
  * dcpp_init_year  (dcpp_init_year) object 8B None
  * lat             (lat) float64 432B 18.5 19.5 20.5 21.5 ... 69.5 70.5 71.5
  * lon             (lon) float64 768B -169.4 -168.1 -166.9 ... -51.88 -50.62
    height          float64 8B ...
Attributes:
    cell_measures:  area: areacella
    cell_methods:   area: time: mean
    interp_method:  conserve_order2
    long_name:      Near-Surface Air Temperature
    original_name:  tas
    standard_name:  air_temperature
    units:          K

In [10]:
YEARS = [2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]

def annual_anomaly_to_df(ds, scenario_name):
    annual = ds["tas"].resample(time="YS").mean()
    anomaly = annual - baseline

    anomaly = anomaly.sel(time=slice("2020-01-01", "2100-12-31"))

    years = anomaly["time"].dt.year
    anomaly = anomaly.sel(time=years.isin(YEARS))

    df = anomaly.to_dataframe(name="temp_anomaly").reset_index()

    df["year"] = [t.year for t in df["time"]]
    df["scenario"] = scenario_name

    df = df[["year", "scenario", "lat", "lon", "temp_anomaly"]].dropna()
    df["lon"] = np.where(df["lon"] > 180, df["lon"] - 360, df["lon"])

    return df

df126 = annual_anomaly_to_df(ssp126_us, "ssp126")
df245 = annual_anomaly_to_df(ssp245_us, "ssp245")
df585 = annual_anomaly_to_df(ssp585_us, "ssp585")

grid_df = pd.concat([df126, df245, df585], ignore_index=True)
grid_df.head()

,year,scenario,lat,lon,temp_anomaly
0,2020,ssp126,18.5,-169.375,0.038269
1,2020,ssp126,18.5,-168.125,0.075531
2,2020,ssp126,18.5,-166.875,0.102325
3,2020,ssp126,18.5,-165.625,0.112213
4,2020,ssp126,18.5,-164.375,0.119232


In [11]:
scenario_labels = {
    "ssp126": "Low emissions",
    "ssp245": "Medium emissions",
    "ssp585": "High emissions"
}

grid_df["scenario_label"] = grid_df["scenario"].map(scenario_labels)

grid_df.head()

,year,scenario,lat,lon,temp_anomaly,scenario_label
0,2020,ssp126,18.5,-169.375,0.038269,Low emissions
1,2020,ssp126,18.5,-168.125,0.075531,Low emissions
2,2020,ssp126,18.5,-166.875,0.102325,Low emissions
3,2020,ssp126,18.5,-165.625,0.112213,Low emissions
4,2020,ssp126,18.5,-164.375,0.119232,Low emissions


In [12]:
states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"

states = gpd.read_file(states_url)

states = states[~states["STUSPS"].isin([
    "PR", "GU", "VI", "MP", "AS", "DC"
])].copy()

states = states[["NAME", "STUSPS", "geometry"]]
states = states.to_crs("EPSG:4326")

states.head()

,NAME,STUSPS,geometry
0,New Mexico,NM,"POLYGON ((-109.05017 31.48, -109.04984 31.4995..."
1,South Dakota,SD,"POLYGON ((-104.05788 44.9976, -104.05078 44.99..."
2,California,CA,"MULTIPOLYGON (((-118.60442 33.47855, -118.5987..."
3,Kentucky,KY,"MULTIPOLYGON (((-89.40565 36.52816, -89.39868 ..."
4,Alabama,AL,"MULTIPOLYGON (((-88.05338 30.50699, -88.05109 ..."


In [13]:
points = gpd.GeoDataFrame(
    grid_df,
    geometry=gpd.points_from_xy(grid_df["lon"], grid_df["lat"]),
    crs="EPSG:4326"
)

joined = gpd.sjoin(
    points,
    states,
    how="inner",
    predicate="within"
)

print("Number of states matched:", joined["NAME"].nunique())

state_check = joined.groupby(["year", "scenario"])["NAME"].nunique().reset_index(name="num_states")
state_check

Number of states matched: 49


,year,scenario,num_states
0,2020,ssp126,49
1,2020,ssp245,49
2,2020,ssp585,49
3,2030,ssp126,49
4,2030,ssp245,49
5,2030,ssp585,49
6,2040,ssp126,49
7,2040,ssp245,49
8,2040,ssp585,49
9,2050,ssp126,49


In [14]:
missing_states = sorted(set(states["NAME"]) - set(joined["NAME"]))
missing_states

['Rhode Island']

In [15]:
unique_points = (
    grid_df[["lat", "lon"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_points_gdf = gpd.GeoDataFrame(
    unique_points,
    geometry=gpd.points_from_xy(unique_points["lon"], unique_points["lat"]),
    crs="EPSG:4326"
)

unique_points_proj = unique_points_gdf.to_crs("EPSG:5070")
states_proj = states.to_crs("EPSG:5070")

missing_polygons = states_proj[states_proj["NAME"].isin(missing_states)].copy()
missing_polygons["geometry"] = missing_polygons.geometry.representative_point()

nearest_missing = gpd.sjoin_nearest(
    missing_polygons,
    unique_points_proj,
    how="left",
    distance_col="distance_to_grid_m"
)

nearest_missing[["NAME", "STUSPS", "lat", "lon", "distance_to_grid_m"]]

,NAME,STUSPS,lat,lon,distance_to_grid_m
41,Rhode Island,RI,41.5,-71.875,28921.483275


In [16]:
missing_rows = []

for _, row in nearest_missing.iterrows():
    state_name = row["NAME"]
    state_abbr = row["STUSPS"]
    lat = row["lat"]
    lon = row["lon"]

    temp = grid_df[
        (grid_df["lat"] == lat) &
        (grid_df["lon"] == lon)
    ].copy()

    temp["NAME"] = state_name
    temp["STUSPS"] = state_abbr

    missing_rows.append(temp)

missing_joined_df = pd.concat(missing_rows, ignore_index=True)

joined_keep = joined.drop(columns=["geometry", "index_right"], errors="ignore")

joined_full = pd.concat(
    [joined_keep, missing_joined_df],
    ignore_index=True
)

print("Number of states matched:", joined_full["NAME"].nunique())

state_check_full = (
    joined_full
    .groupby(["year", "scenario"])["NAME"]
    .nunique()
    .reset_index(name="num_states")
)

state_check_full

Number of states matched: 50


,year,scenario,num_states
0,2020,ssp126,50
1,2020,ssp245,50
2,2020,ssp585,50
3,2030,ssp126,50
4,2030,ssp245,50
5,2030,ssp585,50
6,2040,ssp126,50
7,2040,ssp245,50
8,2040,ssp585,50
9,2050,ssp126,50


In [17]:
state_df = (
    joined_full
    .groupby(["NAME", "STUSPS", "year", "scenario", "scenario_label"])
    .agg(temp_anomaly=("temp_anomaly", "mean"))
    .reset_index()
)

state_df = state_df.rename(columns={
    "NAME": "state",
    "STUSPS": "state_abbr"
})

state_df.head()

,state,state_abbr,year,scenario,scenario_label,temp_anomaly
0,Alabama,AL,2020,ssp126,Low emissions,0.400778
1,Alabama,AL,2020,ssp245,Medium emissions,0.549230
2,Alabama,AL,2020,ssp585,High emissions,2.032163
3,Alabama,AL,2030,ssp126,Low emissions,0.334815
4,Alabama,AL,2030,ssp245,Medium emissions,1.024355


In [18]:
print("Number of states:", state_df["state"].nunique())
print("Rows:", len(state_df))

state_df.groupby(["year", "scenario"])["state"].nunique().reset_index(name="num_states")

Number of states: 50
Rows: 1350


,year,scenario,num_states
0,2020,ssp126,50
1,2020,ssp245,50
2,2020,ssp585,50
3,2030,ssp126,50
4,2030,ssp245,50
5,2030,ssp585,50
6,2040,ssp126,50
7,2040,ssp245,50
8,2040,ssp585,50
9,2050,ssp126,50


In [19]:
q1 = state_df["temp_anomaly"].quantile(1/3)
q2 = state_df["temp_anomaly"].quantile(2/3)

q1, q2

(np.float64(1.284080147743225), np.float64(2.301588137944539))

In [20]:
def add_impact_category(df):
    df = df.copy()

    df["impact_category"] = pd.cut(
        df["temp_anomaly"],
        bins=[-np.inf, q1, q2, np.inf],
        labels=["Low impact", "Medium impact", "High impact"]
    )

    df["impact_score"] = df["impact_category"].map({
        "Low impact": 1,
        "Medium impact": 2,
        "High impact": 3
    }).astype(int)

    return df

state_df = add_impact_category(state_df)

state_df.head()

,state,state_abbr,year,scenario,scenario_label,temp_anomaly,impact_category,impact_score
0,Alabama,AL,2020,ssp126,Low emissions,0.400778,Low impact,1
1,Alabama,AL,2020,ssp245,Medium emissions,0.549230,Low impact,1
2,Alabama,AL,2020,ssp585,High emissions,2.032163,Medium impact,2
3,Alabama,AL,2030,ssp126,Low emissions,0.334815,Low impact,1
4,Alabama,AL,2030,ssp245,Medium emissions,1.024355,Low impact,1


In [21]:
summary = (
    state_df
    .groupby(["year", "scenario", "impact_category"], observed=True)
    .size()
    .reset_index(name="state_count")
)

summary["scenario_label"] = summary["scenario"].map({
    "ssp126": "Low emissions",
    "ssp245": "Medium emissions",
    "ssp585": "High emissions"
})

summary = summary[["year", "scenario", "scenario_label", "impact_category", "state_count"]]

summary

,year,scenario,scenario_label,impact_category,state_count
0,2020,ssp126,Low emissions,Low impact,48
1,2020,ssp126,Low emissions,Medium impact,2
2,2020,ssp245,Medium emissions,Low impact,47
3,2020,ssp245,Medium emissions,Medium impact,3
4,2020,ssp585,High emissions,Low impact,22
...,...,...,...,...,...
62,2100,ssp126,Low emissions,Medium impact,34
63,2100,ssp126,Low emissions,High impact,1
64,2100,ssp245,Medium emissions,Medium impact,8
65,2100,ssp245,Medium emissions,High impact,42


In [22]:
summary.groupby(["year", "scenario"])["state_count"].sum()

year  scenario
2020  ssp126      50
      ssp245      50
      ssp585      50
2030  ssp126      50
      ssp245      50
      ssp585      50
2040  ssp126      50
      ssp245      50
      ssp585      50
2050  ssp126      50
      ssp245      50
      ssp585      50
2060  ssp126      50
      ssp245      50
      ssp585      50
2070  ssp126      50
      ssp245      50
      ssp585      50
2080  ssp126      50
      ssp245      50
      ssp585      50
2090  ssp126      50
      ssp245      50
      ssp585      50
2100  ssp126      50
      ssp245      50
      ssp585      50
Name: state_count, dtype: int64

In [23]:
state_df.isna().sum()

,0
state,0
state_abbr,0
year,0
scenario,0
scenario_label,0
temp_anomaly,0
impact_category,0
impact_score,0


In [24]:
state_df["temp_anomaly"].describe()

,temp_anomaly
count,1350.000000
mean,1.930569
std,1.335777
min,-2.642349
25%,0.989078
50%,1.713003
75%,2.594310
max,6.805716


In [25]:
def find_outliers(group):
    Q1 = group["temp_anomaly"].quantile(0.25)
    Q3 = group["temp_anomaly"].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return group[
        (group["temp_anomaly"] < lower) |
        (group["temp_anomaly"] > upper)
    ]

outliers_grouped = (
    state_df
    .groupby(["year", "scenario"])
    .apply(find_outliers)
)

outliers_grouped.shape

/tmp/ipykernel_54868/1538538833.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(find_outliers)


(41, 8)

In [26]:
impact_scores = {
    "Low impact": 1,
    "Medium impact": 2,
    "High impact": 3
}

state_df["impact_score"] = state_df["impact_category"].map(impact_scores).astype(int)

state_df = state_df[[
    "state",
    "state_abbr",
    "year",
    "scenario",
    "scenario_label",
    "temp_anomaly",
    "impact_category",
    "impact_score"
]].sort_values(["scenario", "year", "state"])

state_df.head()

,state,state_abbr,year,scenario,scenario_label,temp_anomaly,impact_category,impact_score
0,Alabama,AL,2020,ssp126,Low emissions,0.400778,Low impact,1
27,Alaska,AK,2020,ssp126,Low emissions,0.717076,Low impact,1
54,Arizona,AZ,2020,ssp126,Low emissions,-0.840687,Low impact,1
81,Arkansas,AR,2020,ssp126,Low emissions,-0.184711,Low impact,1
108,California,CA,2020,ssp126,Low emissions,0.507053,Low impact,1


In [27]:
category_counts = (
    state_df
    .groupby(["year", "scenario", "impact_category"], observed=True)
    .size()
    .reset_index(name="state_count")
)

category_counts["scenario_label"] = category_counts["scenario"].map({
    "ssp126": "Low emissions",
    "ssp245": "Medium emissions",
    "ssp585": "High emissions"
})

category_counts = category_counts[[
    "year", "scenario", "scenario_label", "impact_category", "state_count"
]]

category_counts.head()

,year,scenario,scenario_label,impact_category,state_count
0,2020,ssp126,Low emissions,Low impact,48
1,2020,ssp126,Low emissions,Medium impact,2
2,2020,ssp245,Medium emissions,Low impact,47
3,2020,ssp245,Medium emissions,Medium impact,3
4,2020,ssp585,High emissions,Low impact,22


In [28]:
category_counts.groupby(["year", "scenario"])["state_count"].sum()

year  scenario
2020  ssp126      50
      ssp245      50
      ssp585      50
2030  ssp126      50
      ssp245      50
      ssp585      50
2040  ssp126      50
      ssp245      50
      ssp585      50
2050  ssp126      50
      ssp245      50
      ssp585      50
2060  ssp126      50
      ssp245      50
      ssp585      50
2070  ssp126      50
      ssp245      50
      ssp585      50
2080  ssp126      50
      ssp245      50
      ssp585      50
2090  ssp126      50
      ssp245      50
      ssp585      50
2100  ssp126      50
      ssp245      50
      ssp585      50
Name: state_count, dtype: int64

In [29]:
print(state_df.shape)
print(category_counts.shape)

print(state_df.isna().sum())

print(category_counts.groupby(["year", "scenario"])["state_count"].sum().unique())

(1350, 8)
(67, 5)
state              0
state_abbr         0
year               0
scenario           0
scenario_label     0
temp_anomaly       0
impact_category    0
impact_score       0
dtype: int64
[50]


In [30]:
state_df.to_csv("us_state_cmip6_impact_categories.csv", index=False)
category_counts.to_csv("us_state_cmip6_category_counts.csv", index=False)

from google.colab import files
files.download("us_state_cmip6_impact_categories.csv")
files.download("us_state_cmip6_category_counts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>